优先级：Scala < SQL、Python、Spark internals、system design.   

如果主力栈是 Python + PySpark，建议按这个优先级来看：
- 第一优先级（直接影响 Spark 面试深度）：第 10 节 Scala + Spark Integration，特别是 Dataset vs DataFrame 的 tradeoff 和 UDF 性能差异——这些在 DE 面试中即使你用 PySpark 答题也可能被追问。
- 第二优先级（FP 思维，体现 senior level）：第 5 节 FP Core，重点是 map vs flatMap 的本质区别、for-comprehension 的展开原理，以及第 4 节 Pattern Matching。
- 第三优先级（如果 JD 明确要求 Scala）：第 3 节 OOP（case class vs trait）、第 6 节错误处理三件套、第 8 节 Implicits。
- 第 11 节的编程题可以作为手感练习，用 functional style 写几遍就能找到 Scala 的 idiomatic 感觉。


Scala 面试速成指南 — DE (NG / Senior) 2026

what is Scala:   
Scala 是一门运行在 Java Virtual Machine (JVM) 上的编程语言，融合了**面向对象（OOP）和函数式编程（FP）**两种范式，常用于数据工程、分布式计算和后端开发。
- Scala = Java 的升级版 + 函数式编程能力
- 和Java一样支持面向对象 + 函数式编程
- 运行在 JVM 上（兼容 Java）
- 相比Java，语法更简洁
  ```java
    // Java
    List<Integer> nums = Arrays.asList(1,2,3);
    nums.stream().map(x -> x * 2).collect(Collectors.toList());
  ```
  ```java
    // Scala
    val nums = List(1,2,3)
    nums.map(_ * 2)
  ```
- 强类型 + 类型推断
  ```java
  val x = 10   // 自动推断为 Int
  ```
- 并发和分布式友好



## 1. 为什么 DE 面试要考 Scala？  

Scala 在 DE 领域的地位来自三个事实：

1. **Spark 是用 Scala 写的**。理解 Scala 意味着你能读 Spark 源码、写高性能 UDF、用 Dataset API 做强类型操作
2. **JVM 生态**。Kafka, Flink, Akka 都是 JVM 系，Scala 与它们的集成比 Python 更自然
3. **FP 思维**。Scala 的 functional programming 范式天然适合分布式数据处理——immutability, pure functions, lazy evaluation 都是分布式系统的核心设计原则

面试定位：大多数 DE 岗位写 "Python or Scala"，但会 Scala 是 senior signal。面试中不会让你手写 type class，但会期望你能用 functional style 解决数据问题、理解 Spark 底层为什么这样设计。

---



## 2. Language Fundamentals（语言基础）

### 2.1 `val` vs `var` vs `def` vs `lazy val`

```java
val x = 10          // immutable binding，赋值后不能改
var y = 10          // mutable binding，可以重新赋值
def z = 10          // 每次访问都重新计算（method，不是变量）
lazy val w = 10     // 第一次访问时才计算，之后缓存结果
```



In [9]:
val x = 11
println(x)
x = 10
println(x)

cmd9.sc:3: reassignment to val
val res9_2 = x = 10
               ^
Compilation Failed

In [8]:
var y = 10
println(y)
y = 11
println(y)

10
11


y: Int = 11

| 关键字 | 可变性 | 求值时机 | 缓存 |
|--------|--------|----------|------|
| `val` | 不可变 | 定义时立即求值 | 是 |
| `var` | 可变 | 定义时立即求值 | 是（但可被覆盖） |
| `def` | — | 每次调用时求值 | 否 |
| `lazy val` | 不可变 | 首次访问时求值 | 是 |

面试高频："**Why prefer `val` over `var`?**"
回答：
- Immutability eliminates race conditions in concurrent and distributed contexts. 
- In Spark, RDDs and Datasets are immutable by design 
  - every transformation creates a new object rather than mutating the existing one, 
  - which enables safe parallel execution and lineage-based fault tolerance.



### 2.2 Type System Basics

Scala 是 **statically typed** 但有 **type inference**，不需要处处写类型：

In [10]:
val name = "hello"         // 推断为 String
val count: Int = 42            // 显式声明
val scores: List[Int] = List(1, 2, 3)

name: String = "hello"
count: Int = 42
scores: List[Int] = List(1, 2, 3)

**Type hierarchy**（面试偶尔考）：
Type Hierarchy（类型层次） 是指 Scala 中所有类型（类、trait、对象）的继承关系和组织结构。理解它能帮助你知道 任意对象可以被赋值给哪些类型、哪些方法可用、以及多态如何工作。

```text
                  Any
                /     \
           AnyVal     AnyRef (= java.lang.Object)
          /  |  \       |   \
       Int Double ...  String  List  ...
                         \      /
                         Null
                           |
                        Nothing
```

- `Any`：所有类型的根
- `AnyVal`：值类型（Int, Double, Boolean...），存在栈上
- `AnyRef`：引用类型（所有 class），存在堆上
- `Nothing`：所有类型的子类型，`throw` 表达式的类型
- `Null`：所有 `AnyRef` 的子类型（但 Scala 鼓励用 `Option` 代替 null）
  - null 没有实例，用于表示 永远不会返回的类型，比如抛异常或无限循环:


In [11]:
def fail(): Nothing = throw new Exception("Oops")

defined function fail

In [ ]:
def func(): Int = ??? //complies, ??? is Nothing

defined function func

In [12]:
// 类型层次使用
val x: Any = 42           // Int 被提升为 Any
val y: AnyRef = "Hello"   // String 是 AnyRef 子类
val z: AnyVal = true      // Boolean 是 AnyVal
val n: Null = null        // null 可以赋值给 AnyRef 类型

x: Any = 42
y: AnyRef = "Hello"
z: AnyVal = true
n: Null = null

### 2.3 String Interpolation
String Interpolation（字符串插值） 是一种简洁、可读性高的方式来构建字符串，它可以直接在字符串里引用变量、表达式或格式化输出，而不需要像 Java 那样用 + 拼接。

Scala 提供三种主要的插值器：  

| 插值器| 功能| 示例| 
| ----| ----| -----| 
| s| 简单插值| $variable 或 ${expression}| 
| f| 格式化插值（类似 printf）| $variable%.2f| 
| raw| 原始插值，不处理转义字符| \n 会原样输出| 


In [13]:
val name = "Scala"
println(s"Hello, $name")                  // s interpolator: 嵌入变量
println(s"1 + 1 = ${1 + 1}")             // 嵌入表达式
println(f"Pi is ${"%.2f".format(3.14159)}") // f interpolator: 格式化
println(raw"No \n escape")                // raw interpolator: 不转义

Hello, Scala
1 + 1 = 2
Pi is 3.14
No \n escape


name: String = "Scala"

In [14]:
// 可以和多行字符串结合使用：
val user = "Yannis"
val text = s"""
  Hello $user,
  Welcome to Scala!
"""
println(text)


  Hello Yannis,
  Welcome to Scala!



user: String = "Yannis"
text: String = """
  Hello Yannis,
  Welcome to Scala!
"""

### 2.4 Expression-Oriented

Scala 里几乎一切都是 **expression**（有返回值），不是 statement.  
- 表达式（expression）会返回一个值，而语句（statement）只是执行动作不返回值。
- Scala 尽量消灭“纯语句”，让代码更函数式、更可组合。


In [15]:
// if 是表达式，有返回值
val x = 10
val result = if (x > 0) "positive" else "non-positive"

x: Int = 10
result: String = "positive"

In [16]:
// match 是表达式
val label = x match {
  case 1 => "one"
  case 2 => "two"
  case _ => "other"
}

label: String = "other"

In [18]:
// 代码块的最后一个表达式就是返回值，不需要 return
def square(x: Int): Int = {
  val result = x * x
  result  // 这就是返回值
}

square(x)

defined function square
res18_1: Int = 100

In [19]:
// 代码块 {} 是表达式例子2，不定义函数
val x = {
  val a = 10
  val b = 20
  a + b   // 最后一行就是返回值
}

x: Int = 30

In [20]:
//函数本身就是表达式
def add1(a: Int, b: Int): Int = a + b
// 或者可以写成
def add2(a: Int, b: Int) = a + b

defined function add1
defined function add2

In [21]:
// try-catch 也是表达式
val result = try {
    val a = 10
    val b = 0
    a/b
} catch {
  case _: Exception => 0
}

result: Int = 0

## 3. OOP Essentials

### 3.1 Class vs Object vs Case Class vs Trait

In [ ]:
// 普通 class
class Person(val name: String, var age: Int) {
  def greet(): String = s"Hi, I'm $name"
}

val Amy = new Person("Amy", 20) 
// 单引号在 Scala 中是 Char 类型
// 普通 class，没有 companion object 提供 apply 方法，所以不能省略 new。
Amy.greet()

defined class Person
Amy: Person = ammonite.$sess.cmd22$Helper$Person@15650a83
res22_2: String = "Hi, I'm Amy"

In [25]:
// object = singleton（单例），也用作 companion object
object Person {
  def apply(name: String): Person = new Person(name, 0) // factory method
}
val p = Person("Alice")  // 不需要 new，调用的是 apply
p.greet()

defined object Person
p: Person = Person(name = "Alice", age = 0)
res25_2: String = "Hi, I'm Alice"

In [ ]:
// case class 自动生成 apply，不用 new
// 自动生成: equals, hashCode, toString, copy, apply, unapply
// 默认参数是 val（不可变）
// 支持 pattern matching
case class User(name: String, age: Int)

val Anna = User("Anna", 25)  // 不需要 new

val u1 = User("Bob", 25)
val u2 = u1.copy(age = 26)   // 不可变更新（u1 本身没有被修改，copy 创建了一个全新的对象 u2）
println(u1 == u2)             // false（structural equality，不是引用比较）


false


defined class User
Anna: User = User(name = "Anna", age = 25)
u1: User = User(name = "Bob", age = 25)
u2: User = User(name = "Bob", age = 26)

🔥 面试高频："**Difference between case class and regular class?**"

| 特性 | `class` | `case class` |
|------|---------|--------------|
| `equals` / `hashCode` | 引用比较（默认） | 结构比较（按字段） |
| `toString` | 类名@hashcode | `User(Bob,25)` |
| `apply` | 需要 `new` | 不需要 `new` |
| `unapply` | 无 | 自动生成（支持 pattern matching） |
| `copy` | 无 | 自动生成 |
| 参数默认 | 需要显式 `val`/`var` | 默认 `val` |
| 用途 | 通用 OOP | **数据载体、Spark schema 定义** |

面试话术："Case classes are Scala's answer to data transfer objects. They give you immutability, structural equality, and pattern matching out of the box — which is why Spark uses them to define strongly-typed Dataset schemas."



在 Scala 中，trait 是一个非常核心的概念，可以理解为： 

👉 “可以包含方法和字段的接口 + mixin（混入）能力”

它比 Java 的 interface 更强大，也是 Scala 支持函数式和面向对象结合的关键。

| 特性| trait| abstract class| 
| ---| ----| ----------------| 
| 是否可多继承| ✅ 可以多个| ❌ 只能一个| 
| 构造函数参数| ❌ 不支持（Scala 3 支持）| ✅ 支持| 
| 用途| 功能组合（mixin）| 基础类| 

trait = 可复用功能模块 + 支持多继承的接口增强版

In [27]:
// trait（类似 Java interface，但可以有实现）
trait Serializable {
  def serialize(): String
}

trait Loggable {
  def log(msg: String): Unit = println(s"[LOG] $msg")
}

// 一个 class 可以 mix in 多个 traits
class Service extends Serializable with Loggable {
  def serialize(): String = "..."
}


defined trait Serializable
defined trait Loggable
defined class Service

### 3.2 Trait vs Abstract Class


In [28]:
abstract class Animal {
  val name: String                    // 抽象字段
  def speak(): String                 // 抽象方法
  def breathe(): String = "inhale"    // 具体方法
}

trait Flyable {
  def fly(): String = "flapping wings"
}

trait Swimmable {
  def swim(): String = "paddling"
}

// 可以 extend 1 个 abstract class + mix in 多个 traits
class Duck extends Animal with Flyable with Swimmable {
  val name = "Duck"
  def speak() = "Quack"
}


defined class Animal
defined trait Flyable
defined trait Swimmable
defined class Duck

| 维度 | `abstract class` | `trait` |
|------|-------------------|---------|
| 继承数量 | 只能 extend 一个 | 可以 mix in 多个 |
| 构造器参数 | 可以有 | Scala 2 不行，**Scala 3 可以** |
| Java 互操作 | 更自然 | 编译为 interface + 静态方法 |
| 典型用途 | 定义 "is-a" 关系 | 定义能力/行为（"has-a"能力） |



### 3.3 Sealed Trait / Sealed Class

在 Scala 中，sealed trait 和 sealed class 是对类型层次的一种限制性设计，核心思想是：

👉 限制继承范围（只能在同一个文件中被继承）


- `sealed` 意味着所有子类必须定义在同一个文件中
- 编译器可以检查 pattern matching 是否穷尽（exhaustive check）
- 非常适合表示 **ADT (Algebraic Data Type)**，在 Spark 自定义类型中很常用



In [ ]:
sealed trait Shape // 所有继承 Shape 的子类 必须写在同一个文件里
case class Circle(radius: Double) extends Shape
case class Rectangle(w: Double, h: Double) extends Shape
case class Triangle(a: Double, b: Double, c: Double) extends Shape

defined trait Shape
defined class Circle
defined class Rectangle
defined class Triangle

In [30]:
import scala.math._
def area(s: Shape): Double = s match {
  case Circle(r) => Pi * r * r
  case Rectangle(w, h) => w * h
  case Triangle(a, b, c) =>
    val p = (a + b + c) / 2
    sqrt(p * (p-a) * (p-b) * (p-c))
  // 编译器会警告如果你漏掉了某个 case
}

val circle1 = Circle(3)
area(circle1)

import scala.math._
defined function area
circle1: Circle = Circle(radius = 3.0)
res30_3: Double = 28.274333882308138

## 4. Pattern Matching（模式匹配）— 面试核心

Pattern matching 是 Scala 最强大的特性之一，远比 Java switch 强大：

### 4.1 基本用法


In [31]:
// 匹配值
x match {
  case 1 => "one"
  case 2 | 3 => "two or three"    // OR pattern
  case n if n > 10 => s"big: $n"  // guard condition
  case _ => "other"               // wildcard（兜底）
}

res31: String = "big: 30"

In [33]:
// 匹配类型
def describe(x: Any): String = x match {
  case i: Int => s"Int: $i"
  case s: String => s"String: $s"
  case l: List[_] => s"List of ${l.length}"
  case _ => "unknown"
}

val x = 34.5
describe(x)

defined function describe
x: Double = 34.5
res33_2: String = "unknown"

### 4.2 Case Class 解构（最常用）


In [34]:
case class Order(id: String, amount: Double, status: String)

def process(order: Order): String = order match {
  case Order(_, amt, _) if amt > 10000 => "needs approval"
  case Order(id, _, "cancelled") => s"order $id was cancelled"
  case Order(_, _, "shipped") => "already shipped"
  case _ => "processing"
}

val order = Order("001", 23, "shipped")
process(order)

defined class Order
defined function process
order: Order = Order(id = "001", amount = 23.0, status = "shipped")
res34_3: String = "already shipped"

### 4.3 Nested Pattern Matching


In [35]:
case class Address(city: String, country: String)
case class Customer(name: String, address: Address)

def greet(c: Customer): String = c match {
  case Customer(name, Address(_, "China")) => s"你好 $name"
  case Customer(name, Address(_, "US")) => s"Hello $name"
  case Customer(name, _) => s"Hi $name"
}

defined class Address
defined class Customer
defined function greet

### 4.4 Collection Pattern Matching
Collection Pattern Matching（集合模式匹配） 是把 match 用在集合（如 List、Seq、Array）上的一种强大用法，可以直接按结构拆解数据。

👉 一句话理解：

不仅匹配值，还可以匹配集合的“形状”（structure）

In [36]:
// 基本示例
// 精确匹配整个集合
val list = List(1, 2, 3)

list match {
  case List(1, 2, 3) => println("exact match")
  case _ => println("something else")
}

exact match


list: List[Int] = List(1, 2, 3)

In [ ]:
// 解构（Destructuring）
// 直接把元素拆出来
val list = List(1, 2, 3)

list match {
    case List(a, b, c) =>
        println(s"a=$a, b=$b, c=$c")
    case _ =>
        println("not a 3-element list")
}

a=1, b=2, c=3


list: List[Int] = List(1, 2, 3)

In [39]:
// Head + Tail（最常用🔥）
val list = List(1, 2, 3, 4)

list match {
  case head :: tail =>
    println(s"head = $head, tail = $tail")
}

cmd39.sc:4: match may not be exhaustive.
It would fail on the following input: Nil
val res39_1 = list match {
              ^


head = 1, tail = List(2, 3, 4)


list: List[Int] = List(1, 2, 3, 4)

:: 表示：
- 左边：第一个元素
- 右边：剩余列表

上面编译器给了一个警告（不是错误）：match may not be exhaustive。
意思是你的 match 只处理了恰好 4 个元素的 List，没有覆盖其他情况：如果 list 是空的（Nil）、只有 1 个元素、有 5 个元素……都会在运行时抛 MatchError。
加一个兜底的 `case _ `就好或者：

In [40]:
val list = List(1)

list match {
  case Nil => "empty"
  case head :: Nil => s"single element: $head"
  case head :: tail => s"head=$head, rest=$tail"
}

list: List[Int] = List(1)
res40_1: String = "single element: 1"

In [42]:
// 多层结构匹配
val list = List(1, 2, 3, 4)

list match {
  case first :: second :: rest =>
    println(s"$first, $second, rest = $rest")
  case _ =>
        println("not a 4-element list")
}

1, 2, rest = List(3, 4)


list: List[Int] = List(1, 2, 3, 4)

In [43]:
// 匹配特定开头
list match {
  case 1 :: rest => println("starts with 1")
  case _ => println("other")
}

starts with 1


In [46]:
// 使用 _ 忽略部分
val list = List(1, 2, 3, 4)
list match {
  case List(_, _, third, _) => println(s"third = $third")
  case _ => println("other")
}

third = 3


list: List[Int] = List(1, 2, 3, 4)

In [48]:
//匹配任意长度（_ *）
list match {
  case List(1, _*) => println("starts with 1")
  case _ => println("other")
}

starts with 1


In [49]:
// Array / Seq 也可以
val arr = Array(1, 2, 3)

arr match {
  case Array(a, b, c) => println(a + b + c)
  case _ => println("other")
}

6


arr: Array[Int] = Array(1, 2, 3)

In [50]:
// Tuple matching
val pair = ("key", 42)
pair match {
  case (k, v) if v > 0 => s"$k -> positive $v"
  case (k, _) => s"$k -> non-positive"
}

pair: (String, Int) = ("key", 42)
res50_1: String = "key -> positive 42"

In [51]:
// 实战例子: 假设你处理日志
val record = List("user1", "click", "2026-03-26")

record match {
  case user :: action :: date :: Nil =>
    println(s"user=$user, action=$action, date=$date")

  case _ =>
    println("invalid record")
}

user=user1, action=click, date=2026-03-26


record: List[String] = List("user1", "click", "2026-03-26")

面试技巧：当面试官让你用 Scala 解决数据问题时，pattern matching + case class 的组合几乎总是最优雅的解法。它展示你理解 Scala 的 idiomatic style。


## 5. Functional Programming Core（FP 核心）

### 5.1 First-Class Functions & Higher-Order Functions
1️⃣ First-Class Functions（函数是一等公民）

函数可以像普通值一样被使用

也就是说函数可以：
- 赋值给变量
- 作为参数传递
- 作为返回值返回

In [ ]:
// 赋值给变量
val add: (Int, Int) => Int = (a, b) => a + b
//or
val add = (a: Int, b: Int) => a + b

add(2, 3)

add: (Int, Int) => Int = ammonite.$sess.cmd52$Helper$$Lambda/0x00001c0001b6c000@1ecd7dde
square: Int => Int = ammonite.$sess.cmd52$Helper$$Lambda/0x00001c0001b67b70@5f578943
res52_2: Int = 5

✅ `(a: Int, b: Int) => a + b`

这是一个匿名函数（lambda）：
- 输入：a, b
- 输出：a + b

✅ `val add = ...`

把这个函数赋值给变量 add

👉 相当于：

add 现在是一个“函数变量”

✅ `add(2, 3)`

调用这个函数: 2 + 3 = 5   

🔥 所以本质上函数和 Int、String 一样，都是“值”

In [59]:
val square: Int => Int = x => x * x

square: Int => Int = ammonite.$sess.cmd59$Helper$$Lambda/0x00001c0001b777b0@2add3e79

In [60]:
// 作为参数传递
def operate(a: Int, b: Int, f: (Int, Int) => Int) = {
  f(a, b)
}

println(operate(2, 3, add))  // 5

5


defined function operate

✅ 函数定义 `def operate(a: Int, b: Int, f: (Int, Int) => Int)`

参数：  
- a, b → 普通整数
- **f → 一个函数**（重点🔥）

👉 f 的类型：`(Int, Int) => Int`     

✅ f(a, b)

调用传进来的函数：f(a, b)
  
✅ 调用时:  `operate(2, 3, add)`   
   
👉 实际发生：

```scala
f = add
→ f(2, 3)
→ 2 + 3
→ 5
```
🔥 本质: 函数可以当“参数”传进去 → 这就是高阶函数的基础

In [58]:
// 作为返回值
def getMultiplier(factor: Int) = {
  println(s"factor = $factor")
  (x: Int) => x * factor
}

val times2 = getMultiplier(2)
println(times2(5))  // 10
println(times2(3))

factor = 2
10
6


defined function getMultiplier
times2: Int => Int = ammonite.$sess.cmd58$Helper$$Lambda/0x00001c0001b76520@3e0cd0df

✅ 定义函数 `def getMultiplier(factor: Int)`   

返回的是一个函数:  
`(x: Int) => x * factor`  

✅ 关键点：闭包（Closure）🔥   

👉 这里的 factor：
- 不是函数参数
- 是“外部变量”

✔ Scala 会“记住”它. 

✅ 调用 

`val times2 = getMultiplier(2)` 等价于 `val times2 = (x: Int) => x * 2`   

🔥 本质: 函数可以“生产函数” + 记住环境（closure）

2️⃣ Higher-Order Functions（高阶函数）

👉 含义：

接受函数作为参数 或 返回函数的函数

In [62]:
val nums = List(1, 2, 3, 4)
// map
nums.map(x => x * 2) // List(2, 4, 6, 8) 
// filter
nums.filter(x => x % 2 == 0) // List(2, 4)
// reduce
nums.reduce((a, b) => a + b) // 10


nums: List[Int] = List(1, 2, 3, 4)
res62_1: List[Int] = List(2, 4, 6, 8)
res62_2: List[Int] = List(2, 4)
res62_3: Int = 10

这些都是高阶函数，因为它们接收函数作为参数.   

简化写法（Scala 很重要🔥）:

In [63]:
nums.map(_ * 2)
nums.filter(_ % 2 == 0)
nums.reduce(_ + _)

res63_0: List[Int] = List(2, 4, 6, 8)
res63_1: List[Int] = List(2, 4)
res63_2: Int = 10

In [64]:
// Higher-order function: 接收函数作为参数
def applyTwice(f: Int => Int, x: Int): Int = f(f(x))
applyTwice(square, 3)  // square(square(3)) = square(9) = 81


defined function applyTwice
res64_1: Int = 81

In [65]:
// 返回函数
def multiplier(factor: Int): Int => Int = x => x * factor
val triple = multiplier(3)
triple(5)  // 15

defined function multiplier
triple: Int => Int = ammonite.$sess.cmd65$Helper$$Lambda/0x00001c0001b7acc8@44790e6f
res65_2: Int = 15

In [66]:
// 假设处理日志
val logs = List(
  ("user1", 100),
  ("user2", 200)
)

// 提取金额并求和
val total = logs.map(_._2).sum

logs: List[(String, Int)] = List(("user1", 100), ("user2", 200))
total: Int = 300

Tuple（元组）访问方式
- _1 → 第一个元素 "user1"
- _2 → 第二个元素 100  

`_._2` :  对每个元素，取它的第2个值

### 5.2 核心集合操作（面试必考）



In [ ]:
val nums = List(1, 2, 3, 4, 5)

// map: 对每个元素应用函数
nums.map(_ * 2)               // List(2, 4, 6, 8, 10)

// filter: 保留满足条件的元素
nums.filter(_ > 3)            // List(4, 5)

// flatMap: map + flatten（最重要的操作之一）
val words = List("hello world", "hi there")
words.map(_.split(" "))       // List(Array(hello, world), Array(hi, there))
words.flatMap(_.split(" "))   // List(hello, world, hi, there) ← 拍平了

// fold / reduce: 聚合
nums.foldLeft(0)(_ + _)       // 15  (初始值 0，从左到右累加)
nums.reduce(_ + _)            // 15  (无初始值，要求非空)



fold 的本质：

把一个集合“折叠”成一个值

- foldLeft(initial)(function)
- foldRight(initial)(function)

In [71]:
val nums = List(1, 2, 3, 4)
// foldLeft vs foldRight
nums.foldLeft("")((acc, n) => acc + n.toString)   // "1234"
nums.foldRight("")((n, acc) => acc + n.toString)  // "4321"

nums: List[Int] = List(1, 2, 3, 4)
res71_1: String = "1234"
res71_2: String = "4321"

```scala 
nums.foldLeft("")((acc, n) => acc + n.toString)
```
执行顺序:  

```text
初始: acc = ""

Step1: acc = "" + "1"   → "1"
Step2: acc = "1" + "2"  → "12"
Step3: acc = "12" + "3" → "123"
Step4: acc = "123" + "4" → "1234"
```

`nums.foldRight("")((acc, n) => acc + n.toString)`为什么会错.  
  - 参数顺序写反了
  - 函数签名: foldRight[B](z: B)(op: (A, B) => B)
    - A = 集合元素类型（这里是 Int）
    - B = 累加器类型（这里是 String）

In [72]:
// groupBy: 分组（MapReduce 思维！）
val data = List(("a", 1), ("b", 2), ("a", 3))
data.groupBy(_._1)            // Map("a" -> List(("a",1),("a",3)), "b" -> List(("b",2)))

data: List[(String, Int)] = List(("a", 1), ("b", 2), ("a", 3))
res72_1: Map[String, List[(String, Int)]] = HashMap(
  "a" -> List(("a", 1), ("a", 3)),
  "b" -> List(("b", 2))
)

In [ ]:
val nums = List(1, 2, 3, 4, 5)
// collect: filter + map 的组合（用 partial function）
nums.collect { case x if x % 2 == 0 => x * 10 }  // List(20, 40)

// partition: 一分为二
val (evens, odds) = nums.partition(_ % 2 == 0)  // (List(2,4), List(1,3,5))

nums: List[Int] = List(1, 2, 3, 4, 5)
res74_1: List[Int] = List(20, 40)
evens: List[Int] = List(2, 4)
odds: List[Int] = List(1, 3, 5)

In [75]:
// zip: 配对
List("a","b","c").zip(List(1,2,3))  // List(("a",1), ("b",2), ("c",3))

res75: List[(String, Int)] = List(("a", 1), ("b", 2), ("c", 3))

### 5.3 `map` vs `flatMap` 深度理解

这是面试中区分初级和高级 Scala 程序员的关键：

In [76]:
// map 保持结构：List[A] => List[B]
List(1, 2, 3).map(x => List(x, x*10))
// List(List(1, 10), List(2, 20), List(3, 30))  ← 嵌套了

// flatMap 拍平一层：List[A] => List[B]
List(1, 2, 3).flatMap(x => List(x, x*10))
// List(1, 10, 2, 20, 3, 30)  ← 平的

res76_0: List[List[Int]] = List(List(1, 10), List(2, 20), List(3, 30))
res76_1: List[Int] = List(1, 10, 2, 20, 3, 30)

In [77]:
// flatMap 在 Option 中特别有用
def findUser(id: Int): Option[String] = if (id == 1) Some("Alice") else None
def findEmail(name: String): Option[String] = if (name == "Alice") Some("alice@x.com") else None

// 链式调用，任何一步返回 None 就短路
findUser(1).flatMap(findEmail)  // Some("alice@x.com")
findUser(2).flatMap(findEmail)  // None

// 等价的 for-comprehension（语法糖）
for {
  name  <- findUser(1)
  email <- findEmail(name)
} yield email  // Some("alice@x.com")

defined function findUser
defined function findEmail
res77_2: Option[String] = Some(value = "alice@x.com")
res77_3: Option[String] = None
res77_4: Option[String] = Some(value = "alice@x.com")

面试话术：
- flatMap is the workhorse of monadic composition in Scala. 
- In the context of Spark, flatMap on an RDD is equivalent to the map phase of MapReduce 
  - each input element can produce zero or more output elements, 
- and the results are flattened into a single collection.



### 5.4 For-Comprehension（语法糖）

For-comprehension 本质上是 `map`, `flatMap`, `filter` 的语法糖：


In [78]:
// 这段 for-comprehension:
for {
  x <- List(1, 2, 3)
  y <- List(10, 20)
  if x + y > 11
} yield x * y

// 编译器会翻译成:
List(1, 2, 3)
  .flatMap(x => List(10, 20)
    .withFilter(y => x + y > 11)
    .map(y => x * y))

// 结果: List(20, 20, 40, 30, 60)

res78_0: List[Int] = List(20, 20, 40, 30, 60)
res78_1: List[Int] = List(20, 20, 40, 30, 60)

### 5.5 Partial Functions


In [79]:
// PartialFunction: 只对部分输入有定义
val handler: PartialFunction[Int, String] = {
  case 200 => "OK"
  case 404 => "Not Found"
  case 500 => "Internal Server Error"
}

handler.isDefinedAt(200)  // true
handler.isDefinedAt(301)  // false

// 常用在 collect 中
List(200, 301, 404, 500).collect(handler)  // List("OK", "Not Found", "Internal Server Error")


handler: PartialFunction[Int, String] = <function1>
res79_1: Boolean = true
res79_2: Boolean = false
res79_3: List[String] = List("OK", "Not Found", "Internal Server Error")

只处理了部分情况，忽略301: 

In [80]:
handler.isDefinedAt(301) // false
handler.isDefinedAt(200) // true

res80_0: Boolean = false
res80_1: Boolean = true

### 5.6 Currying & Partially Applied Functions


In [81]:
// Currying: 多参数函数 → 链式单参数函数
def add(a: Int)(b: Int): Int = a + b
val add5 = add(5) _        // Int => Int
add5(3)                     // 8

// 实际用途: 类型推断 + API 设计
def transform[A, B](list: List[A])(f: A => B): List[B] = list.map(f)
transform(List(1, 2, 3))(x => x * 2)  // 编译器可以从第一个参数推断 A = Int

defined function add
add5: Int => Int = ammonite.$sess.cmd81$Helper$$Lambda/0x00001c0001b9ba38@31afcced
res81_2: Int = 8
defined function transform
res81_4: List[Int] = List(2, 4, 6)

## 6. Option, Either, Try（错误处理三件套）

### 6.1 Option[T]（替代 null）


In [88]:
// Option 有两个子类: Some(value) 和 None
val maybeUser: Option[String] = Some("Alice")
val noUser: Option[String] = None

// 安全访问
maybeUser.getOrElse("Unknown")  // "Alice"
noUser.getOrElse("Unknown")     // "Unknown"

// 千万不要用 .get（会抛 NoSuchElementException）
// maybeUser.get  // 危险！

// Option 是 collection-like，支持 map/flatMap/filter
maybeUser.map(_.toUpperCase)       // Some("ALICE")
noUser.map(_.toUpperCase)          // None（不会 NPE！）

maybeUser: Option[String] = Some(value = "Alice")
noUser: Option[String] = None
res88_2: String = "Alice"
res88_3: String = "Unknown"
res88_4: Option[String] = Some(value = "ALICE")
res88_5: Option[String] = None

In [89]:
// 链式操作
def findById(id: Int): Option[User] = ???
def getAddress(u: User): Option[Address] = ???
def getZip(a: Address): Option[String] = ???

val zip: Option[String] = for {
  user    <- findById(1)
  address <- getAddress(user)
  zip     <- getZip(address)
} yield zip
// 任何一步是 None，整个结果就是 None，不会 NPE

scala.NotImplementedError: an implementation is missing

面试话术： 
- Option forces you to handle the absence case at compile time rather than discovering NullPointerExceptions at runtime. 
- This is especially important in data pipelines where null values in one stage can cascade into silent data corruption downstream."



In [90]:
case class User(id: Int, name: String)
case class Address(street: String, zip: String)

def findById(id: Int): Option[User] = 
  if (id == 1) Some(User(1, "Alice")) else None

def getAddress(u: User): Option[Address] = 
  Some(Address("123 Main St", "98101"))

def getZip(a: Address): Option[String] = 
  Some(a.zip)

val zip: Option[String] = for {
  user    <- findById(1)
  address <- getAddress(user)
  zip     <- getZip(address)
} yield zip

// 输出: Some(98101)
println(zip)

Some(98101)


defined class User
defined class Address
defined function findById
defined function getAddress
defined function getZip
zip: Option[String] = Some(value = "98101")

### 6.2 Either[L, R]（有错误信息的结果）



In [91]:
// Either: Left = 错误, Right = 成功（by convention）
def divide(a: Int, b: Int): Either[String, Double] =
  if (b == 0) Left("Division by zero")
  else Right(a.toDouble / b)

divide(10, 3)  // Right(3.333...)
divide(10, 0)  // Left("Division by zero")

defined function divide
res91_1: Either[String, Double] = Right(value = 3.3333333333333335)
res91_2: Either[String, Double] = Left(value = "Division by zero")

In [92]:
// Right-biased: map/flatMap 只作用于 Right
divide(10, 3).map(_ * 2)        // Right(6.666...)
divide(10, 0).map(_ * 2)        // Left("Division by zero")

res92_0: Either[String, Double] = Right(value = 6.666666666666667)
res92_1: Either[String, Double] = Left(value = "Division by zero")

In [94]:
// Pattern matching
divide(10, 3) match {
  case Right(result) => println(s"Result: $result")
  case Left(error)   => println(s"Error: $error")
}

divide(10, 0) match {
  case Right(result) => println(s"Result: $result")
  case Left(error)   => println(s"Error: $error")
}

Result: 3.3333333333333335
Error: Division by zero


### 6.3 Try[T]（捕获异常）


In [95]:
import scala.util.{Try, Success, Failure}

// Try 自动捕获异常
val result: Try[Int] = Try("123".toInt)     // Success(123)
val failed: Try[Int] = Try("abc".toInt)     // Failure(NumberFormatException)

// 链式操作（和 Option 一样）
Try("42".toInt)
  .map(_ * 2)
  .getOrElse(0)  // 84

// 在数据处理中很实用：跳过脏数据
val rawData = List("1", "abc", "3", "def", "5")
val cleaned = rawData.flatMap(s => Try(s.toInt).toOption)  // List(1, 3, 5)

import scala.util.{Try, Success, Failure}
result: Try[Int] = Success(value = 123)
failed: Try[Int] = Failure(
  exception = java.lang.NumberFormatException: For input string: "abc"
)
res95_3: Int = 84
rawData: List[String] = List("1", "abc", "3", "def", "5")
cleaned: List[Int] = List(1, 3, 5)

### 6.4 三者对比

| 类型 | 表达 | 适用场景 |
|------|------|----------|
| `Option[T]` | 有值 / 无值 | 查找、可选字段 |
| `Either[E, T]` | 错误信息 / 成功值 | 需要知道失败原因 |
| `Try[T]` | 异常 / 成功值 | 可能抛异常的操作 |

在 Spark 中，处理脏数据时 `Try` + `flatMap` 组合非常常用——把解析失败的行安全地过滤掉，而不是让整个 job crash。

## 7. Collections Deep Dive

### 7.1 Immutable vs Mutable


In [97]:
// 默认导入的是 immutable
import scala.collection.immutable  // 默认
import scala.collection.mutable    // 需要显式导入

val immList = immutable.List(1, 2, 3)        // 不可变
val mutList = mutable.ListBuffer(1, 2, 3)  // 可变
mutList += 4                        // ListBuffer(1, 2, 3, 4)

import scala.collection.immutable
import scala.collection.mutable
immList: List[Int] = List(1, 2, 3)
mutList: mutable.ListBuffer[Int] = ListBuffer(1, 2, 3, 4)
res97_4: mutable.ListBuffer[Int] = ListBuffer(1, 2, 3, 4)

### 7.2 常用集合一览

| 类型 | 特点 | 典型用法 |
|------|------|----------|
| `List` | 链表，prepend O(1), append O(n) | 函数式递归处理 |
| `Vector` | 树形，随机访问 O(log32 n) | 通用默认选择 |
| `Array` | 底层 Java 数组，随机访问 O(1) | 性能敏感场景 |
| `Set` | 去重集合 | 唯一性检查 |
| `Map` | 键值对 | lookup / grouping |
| `Seq` | 有序序列（抽象） | 通常用 `List` 或 `Vector` |
| `Stream` / `LazyList` | 惰性求值的无限序列 | 按需生成数据 |
| `Tuple` | 固定长度异构 | Spark PairRDD 的 (K, V) |



### 7.3 集合操作 Cheat Sheet


In [100]:
val xs = List(3, 1, 4, 1, 5, 9, 2, 6)

xs.sorted                    // List(1, 1, 2, 3, 4, 5, 6, 9)
xs.sortBy(-_)                // List(9, 6, 5, 4, 3, 2, 1, 1) 降序
xs.distinct                  // List(3, 1, 4, 5, 9, 2, 6)
xs.take(3)                   // List(3, 1, 4)
xs.drop(3)                   // List(1, 5, 9, 2, 6)
xs.takeWhile(_ < 5)          // List(3, 1, 4, 1)
xs.sliding(3).toList          // 滑动窗口: List(List(3,1,4), List(1,4,1), ...)
xs.grouped(3).toList          // 分组: List(List(3,1,4), List(1,5,9), List(2,6))
xs.mkString(", ")             // "3, 1, 4, 1, 5, 9, 2, 6"

// Map 操作
val m = Map("a" -> 1, "b" -> 2)
m.get("a")                   // Some(1)
m.get("z")                   // None
m.getOrElse("z", 0)          // 0
m + ("c" -> 3)               // Map(a->1, b->2, c->3) 新 Map
m.view.mapValues(_ * 10)           // Map(a->10, b->20)
m.view.filterKeys(_ != "a")       // Map(b->2)


xs: List[Int] = List(3, 1, 4, 1, 5, 9, 2, 6)
res100_1: List[Int] = List(1, 1, 2, 3, 4, 5, 6, 9)
res100_2: List[Int] = List(9, 6, 5, 4, 3, 2, 1, 1)
res100_3: List[Int] = List(3, 1, 4, 5, 9, 2, 6)
res100_4: List[Int] = List(3, 1, 4)
res100_5: List[Int] = List(1, 5, 9, 2, 6)
res100_6: List[Int] = List(3, 1, 4, 1)
res100_7: List[List[Int]] = List(
  List(3, 1, 4),
  List(1, 4, 1),
  List(4, 1, 5),
  List(1, 5, 9),
  List(5, 9, 2),
  List(9, 2, 6)
)
res100_8: List[List[Int]] = List(List(3, 1, 4), List(1, 5, 9), List(2, 6))
res100_9: String = "3, 1, 4, 1, 5, 9, 2, 6"
m: Map[String, Int] = Map("a" -> 1, "b" -> 2)
res100_11: Option[Int] = Some(value = 1)
res100_12: Option[Int] = None
res100_13: Int = 0
res100_14: Map[String, Int] = Map("a" -> 1, "b" -> 2, "c" -> 3)
res100_15: collection.MapView[String, Int] = MapView(("a", 10), ("b", 20))
res100_16: collection.MapView[String, Int] = MapView(("b", 2))

### 7.4 经典面试题：用集合操作实现 WordCount



In [104]:
val text = "hello world hello scala world hello"

// 方法 1: groupBy + mapValues
text.split(" ")
  .groupBy(identity)        // Map("hello" -> Array("hello","hello","hello"), ...)
  .view.mapValues(_.length)       // Map("hello" -> 3, "world" -> 2, "scala" -> 1)



text: String = "hello world hello scala world hello"
res104_1: collection.MapView[String, Int] = MapView(
  ("world", 2),
  ("scala", 1),
  ("hello", 3)
)

In [102]:
// 方法 2: foldLeft
text.split(" ")
  .foldLeft(Map.empty[String, Int]) { (acc, word) =>
    acc + (word -> (acc.getOrElse(word, 0) + 1))
  }



res102: Map[String, Int] = Map("hello" -> 3, "world" -> 2, "scala" -> 1)

In [103]:
// 方法 3: groupMapReduce (Scala 2.13+)
text.split(" ")
  .groupMapReduce(identity)(_ => 1)(_ + _)


res103: Map[String, Int] = Map("world" -> 2, "scala" -> 1, "hello" -> 3)

## 8. Implicits（隐式机制）— Senior 必知

### 8.1 Implicit Parameters
Implicit Parameters = 编译器会自动帮你传的参数

换句话说，你可以：
- 定义一个参数为 implicit
- 调用函数时不显式传值
- Scala 编译器会自动在作用域中找匹配的值

In [105]:
// 定义一个隐式值
implicit val defaultTimeout: Int = 30

// 函数声明隐式参数
def fetchData(url: String)(implicit timeout: Int): String =
  s"Fetching $url with timeout $timeout"

fetchData("https://api.example.com")  // 自动使用 30
fetchData("https://api.example.com")(60)  // 显式覆盖

defaultTimeout: Int = 30
defined function fetchData
res105_2: String = "Fetching https://api.example.com with timeout 30"
res105_3: String = "Fetching https://api.example.com with timeout 60"

### 8.2 Implicit Conversions

> ⚠️ Scala 3 已经用 `given` / `using` / `extension` 替代了大部分 implicit 用法。但面试中 Scala 2 的 implicit 仍然是高频考点。

In [107]:
// 自动类型转换
import scala.language.implicitConversions

implicit def intToString(x: Int): String = x.toString

val s: String = 42  // 自动调用 intToString(42)

import scala.language.implicitConversions
defined function intToString
s: String = "42"

### 8.3 在 Spark 中的实际应用


In [ ]:
import spark.implicits._  // 这行到底做了什么？

// 它导入了一组 implicit conversions，让你能够:
// 1. 把基本类型的 RDD 转成 DataFrame
val rdd = sc.parallelize(Seq(1, 2, 3))
rdd.toDF("number")  // 需要 implicits._

// 2. 把 case class 的集合转成 Dataset
case class Person(name: String, age: Int)
val ds = Seq(Person("Alice", 30)).toDS()  // 需要 implicits._

// 3. 用 $ 符号引用列
df.select($"name", $"age" + 1)  // $ 是一个 implicit conversion

面试话术:
- When you import `spark.implicits._`, you're bringing in **a set of Encoders** and implicit conversions that bridge Scala's type system with **Spark's Catalyst** type system. 
- The Encoders tell Spark how to **serialize/deserialize** JVM objects to its internal **tungsten binary format**."

  - Catalyst：负责 SQL/DataFrame 的查询优化和物理计划生成
  - Tungsten：负责 内存布局优化 + CPU 执行优化
  - Tungsten Binary Format：Catalyst 优化后的数据最终在内存中的高性能二进制存储方式

### 8.4 Scala 3 的替代（了解即可）
> Almond kernel 跑的是 Scala 2，不认识这些关键字

In [ ]:
// Scala 3: given + using 替代 implicit
given defaultTimeout: Int = 30

def fetchData(url: String)(using timeout: Int): String =
  s"Fetching $url with timeout $timeout"

// Scala 3: extension methods 替代 implicit class
extension (s: String)
  def greet: String = s"Hello, $s!"

"World".greet  // "Hello, World!"

## 9. Concurrency Basics

### 9.1 Future


In [ ]:
import scala.concurrent.{Future, Await}
import scala.concurrent.ExecutionContext.Implicits.global
import scala.concurrent.duration._

// Future: 异步计算的容器
val futureResult: Future[Int] = Future {
  Thread.sleep(1000)
  42
}

// 非阻塞组合（推荐）
futureResult.map(_ * 2)           // Future[Int] = 84
futureResult.flatMap(x => Future(x + 1))

// 多个 Future 并行执行
val f1 = Future { fetchFromDB() }
val f2 = Future { callAPI() }
val f3 = Future { readFile() }

// for-comprehension 组合结果
val combined: Future[(Data, Response, String)] = for {
  db   <- f1
  api  <- f2
  file <- f3
} yield (db, api, file)

// 阻塞等待（仅测试用，生产中避免）
val result = Await.result(combined, 5.seconds)

// 错误处理
futureResult.recover {
  case _: TimeoutException => -1
  case _: Exception => 0
}

### 9.2 面试中的考察方式

通常不会深考 Akka 或 ZIO，但会问：

**How does Future relate to Spark's execution model?**   
- Spark 的 driver 向 executors 提交 tasks 本质上是异步的，
- 但 Spark 自己管理了这层并发，用户不需要直接操作 Future

**What's the difference between concurrency and parallelism?**  
- Concurrency 是结构上的（多个任务交替执行），
- parallelism 是执行上的（多个任务同时执行）。
- Scala Future 提供 concurrency，JVM 线程池提供 parallelism


## 10. Scala + Spark Integration（面试核心）

### 10.1 Case Class → Dataset Schema


In [ ]:
case class Event(
  userId: String,
  eventType: String,
  timestamp: Long,
  value: Double
)

// case class 自动映射为 Spark schema
val events: Dataset[Event] = spark.read
  .json("/data/events")
  .as[Event]  // 强类型 Dataset

// 类型安全的操作
events
  .filter(_.eventType == "purchase")
  .map(e => (e.userId, e.value))
  .groupByKey(_._1)
  .mapValues(_._2)
  .reduceGroups(_ + _)

### 10.2 Dataset (Typed) vs DataFrame (Untyped)


In [ ]:
// DataFrame = Dataset[Row]，弱类型
val df: DataFrame = spark.read.json("/data/events")
df.filter($"eventType" === "purchase")  // 运行时才发现列名错误

// Dataset[Event]，强类型
val ds: Dataset[Event] = df.as[Event]
ds.filter(_.eventType == "purchase")  // 编译时就能检查

| 维度 | DataFrame (`Dataset[Row]`) | Dataset[T] |
|------|---------------------------|------------|
| 类型检查 | 运行时 | 编译时 |
| API | Column-based (`$"col"`) | Lambda-based (`.map`, `.filter`) |
| Catalyst 优化 | 完全优化 | 部分操作无法优化（lambda 是黑盒） |
| 序列化 | Tungsten binary | 需要 Encoder |
| 适用场景 | SQL-like 分析、ETL | 复杂业务逻辑、类型安全 |

面试话术:  
- DataFrames give you the best of **Catalyst optimization** 
  - because the optimizer can see inside Column expressions. 
- Datasets with lambdas are **type-safe but opaque to the optimizer** 
  - Catalyst can't push predicates into a lambda. 
- In practice, I default to DataFrame API and switch to Dataset only when the business logic is complex enough that compile-time type safety outweighs the optimization loss.



### 10.3 UDF in Scala vs PySpark


In [ ]:
// Scala UDF: 直接在 JVM 内执行
import org.apache.spark.sql.functions.udf

val normalizeEmail = udf((email: String) =>
  Option(email).map(_.trim.toLowerCase).getOrElse("")
)

df.withColumn("clean_email", normalizeEmail($"email"))

**为什么 Scala UDF 比 PySpark UDF 快？**

```text
PySpark UDF 执行路径:
JVM (Spark) → 序列化数据 → Python 进程 → 执行 UDF → 序列化结果 → JVM

Scala UDF 执行路径:
JVM (Spark) → 执行 UDF → 完成
```

- PySpark UDF 每行数据要跨进程序列化/反序列化两次
- Scala UDF 在同一个 JVM 进程内直接执行，零序列化开销
- PySpark 的 Pandas UDF (Arrow-based) 改善了这个问题（批量序列化），但仍不如 Scala native



### 10.4 Broadcast + Map-Side Join




In [ ]:
import org.apache.spark.sql.functions.broadcast

// 小表 broadcast 到所有 executor
val smallDF = spark.read.parquet("/data/dim_country")
val largeDF = spark.read.parquet("/data/fact_orders")

// Broadcast join：避免 shuffle
largeDF.join(broadcast(smallDF), Seq("country_code"))

### 10.5 Accumulator & Broadcast Variable


In [ ]:
// Accumulator: 只能 add 的共享变量（用于 counter / metrics）
val errorCount = sc.longAccumulator("error_count")

rdd.foreach { record =>
  if (record.contains("ERROR")) errorCount.add(1)
}
println(s"Total errors: ${errorCount.value}")

// Broadcast Variable: 只读的共享数据（广播到每个 executor）
val lookup = sc.broadcast(Map("US" -> "United States", "CN" -> "China"))

rdd.map(code => lookup.value.getOrElse(code, "Unknown"))

## 11. Classic Interview Coding Problems

### 11.1 FizzBuzz (Functional Style)



In [110]:
(1 to 100).map {
  case n if n % 15 == 0 => "FizzBuzz"
  case n if n % 3 == 0  => "Fizz"
  case n if n % 5 == 0  => "Buzz"
  case n                 => n.toString
}.foreach(println)

1
2
Fizz
4
Buzz
Fizz
7
8
Fizz
Buzz
11
Fizz
13
14
FizzBuzz
16
17
Fizz
19
Buzz
Fizz
22
23
Fizz
Buzz
26
Fizz
28
29
FizzBuzz
31
32
Fizz
34
Buzz
Fizz
37
38
Fizz
Buzz
41
Fizz
43
44
FizzBuzz
46
47
Fizz
49
Buzz
Fizz
52
53
Fizz
Buzz
56
Fizz
58
59
FizzBuzz
61
62
Fizz
64
Buzz
Fizz
67
68
Fizz
Buzz
71
Fizz
73
74
FizzBuzz
76
77
Fizz
79
Buzz
Fizz
82
83
Fizz
Buzz
86
Fizz
88
89
FizzBuzz
91
92
Fizz
94
Buzz
Fizz
97
98
Fizz
Buzz


### 11.2 Flatten Nested List


In [111]:
def flatten(xs: List[Any]): List[Any] = xs match {
  case Nil => Nil
  case (head: List[_]) :: tail => flatten(head) ::: flatten(tail)
  case head :: tail => head :: flatten(tail)
}

flatten(List(1, List(2, 3), List(4, List(5, 6))))
// List(1, 2, 3, 4, 5, 6)

defined function flatten
res111_1: List[Any] = List(1, 2, 3, 4, 5, 6)

### 11.3 Group Consecutive Duplicates




In [112]:
def pack[A](xs: List[A]): List[List[A]] = xs match {
  case Nil => Nil
  case _ =>
    val (same, rest) = xs.span(_ == xs.head)
    same :: pack(rest)
}

pack(List(1, 1, 2, 3, 3, 3, 2, 2))
// List(List(1, 1), List(2), List(3, 3, 3), List(2, 2))

defined function pack
res112_1: List[List[Int]] = List(List(1, 1), List(2), List(3, 3, 3), List(2, 2))

### 11.4 Implement Map Using FoldLeft


In [113]:
def myMap[A, B](xs: List[A])(f: A => B): List[B] =
  xs.foldLeft(List.empty[B])((acc, x) => acc :+ f(x))

// 更高效的版本（用 reverse 避免 :+ 的 O(n) 开销）
def myMapFast[A, B](xs: List[A])(f: A => B): List[B] =
  xs.foldLeft(List.empty[B])((acc, x) => f(x) :: acc).reverse

defined function myMap
defined function myMapFast

### 11.5 Two Sum (Functional Style)


In [117]:
def twoSum(nums: List[Int], target: Int): Option[(Int, Int)] = {
  nums.indices.view
    .flatMap(i => nums.indices.drop(i + 1).map(j => (i, j)))
    .find { case (i, j) => nums(i) + nums(j) == target }
}

// 更高效: 用 foldLeft + Map
def twoSumFast(nums: List[Int], target: Int): Option[(Int, Int)] = {
  nums.zipWithIndex.foldLeft((Map.empty[Int, Int], Option.empty[(Int, Int)])) {
    case ((seen, None), (num, idx)) =>
      seen.get(target - num) match {
        case Some(prevIdx) => (seen, Some((prevIdx, idx)))
        case None => (seen + (num -> idx), None)
      }
    case (result, _) => result
  }._2
}

twoSumFast(List(1, 2, 3, 4, 5), 5)

defined function twoSum
defined function twoSumFast
res117_2: Option[(Int, Int)] = Some(value = (1, 2))

### 11.6 Word Frequency with Sorting



In [116]:
val text = "the quick brown fox jumps over the lazy dog the fox"

text.split(" ")
  .groupBy(identity)
  .view.mapValues(_.length)
  .toList
  .sortBy(-_._2)  // 按频率降序
// List((the,3), (fox,2), (quick,1), (brown,1), ...)

text: String = "the quick brown fox jumps over the lazy dog the fox"
res116_1: List[(String, Int)] = List(
  ("the", 3),
  ("fox", 2),
  ("lazy", 1),
  ("dog", 1),
  ("over", 1),
  ("brown", 1),
  ("quick", 1),
  ("jumps", 1)
)

## 12. Scala 2 vs Scala 3 Key Differences（了解即可）

| 特性 | Scala 2 | Scala 3 |
|------|---------|---------|
| Implicits | `implicit val/def/class` | `given`, `using`, `extension` |
| 枚举 | `sealed trait` + `case object` | 原生 `enum` |
| Union Types | 不支持 | `Int \| String` |
| 语法 | 大括号 + `=>` | 可选缩进语法（类 Python） |
| Trait 参数 | 不支持 | 支持 |
| Null Safety | `Option` (convention) | 实验性 explicit nulls |
| 宏 | `scala.reflect` (复杂) | `inline` + `quotes` |

> 面试中除非明确要求 Scala 3，否则回答 Scala 2 的语法即可。大部分生产环境 Spark 集群还在跑 Scala 2.12/2.13。

## 13. Quick-Fire Q&A（高频面试问答）

### Q: What is tail recursion and why does it matter?
**A:** Tail recursion 是指递归调用是函数的最后一个操作。Scala 编译器会把 tail-recursive 函数优化成 loop，避免 stack overflow。用 `@tailrec` 注解让编译器检查：


In [ ]:
import scala.annotation.tailrec

@tailrec
def factorial(n: Int, acc: Int = 1): Int =
  if (n <= 1) acc
  else factorial(n - 1, n * acc)  // tail position ✓

### Q: What's the difference between `==` and `eq`?
**A:** `==` 是 structural equality（调用 `equals` 方法），`eq` 是 reference equality（同一个对象）。Case class 重写了 `equals`，所以 `User("A", 1) == User("A", 1)` 是 true，但 `eq` 是 false。



### Q: Explain Scala's `Nothing` and `Null`.
**A:** `Nothing` 是所有类型的子类型（bottom type），没有实例。`throw` 的返回类型就是 `Nothing`。`Null` 是所有引用类型的子类型，只有一个实例 `null`。在类型安全的 Scala 代码中，应该用 `Option` 代替 `null`。



### Q: What is an Encoder in Spark?
**A:** Encoder 告诉 Spark 如何在 JVM 对象和 Spark 的内部 Tungsten binary format 之间序列化/反序列化。Case class 的 Encoder 通过 `spark.implicits._` 自动生成。自定义类型需要显式提供 Encoder，否则无法创建 Dataset。



### Q: Why is Scala preferred for Spark over Java?
**A:**
- 更简洁的语法（同样逻辑 Java 要 5x 代码量）
- 原生 FP 支持（map/flatMap/filter 链式调用）
- Case class 无缝映射 Spark schema
- Pattern matching 简化复杂数据处理逻辑
- Spark API 本身是用 Scala 设计的，Scala 能用到所有最新特性



### Q: When would you choose PySpark over Scala Spark?
**A:**
- 团队成员大多数是 Python 背景
- 需要用到 Python ML 生态（scikit-learn, TensorFlow, PyTorch）
- 快速原型rapid prototyping开发（Python REPL + notebooks 更方便）
- 数据分析 + 可视化（pandas, matplotlib 生态更成熟）

但如果是Performance sensitive pipeline，Scala 的优势在于 **无 serialization overhead** + **编译时类型安全** + **更好的 Catalyst 集成**。

---



## 14. Terminology Cheat Sheet

| English Term | 中文 | 一句话解释 |
|---|---|---|
| Immutability | 不可变性 | 赋值后不能修改，`val` 的核心特性 |
| Case Class | 样例类 | 自动生成 equals/hashCode/copy 的数据类 |
| Pattern Matching | 模式匹配 | 比 switch 强大的结构化匹配 |
| Trait | 特质 | 可 mix in 的接口（可含实现） |
| Higher-Order Function | 高阶函数 | 接受或返回函数的函数 |
| Partial Function | 偏函数 | 只对部分输入有定义的函数 |
| Currying | 柯里化 | 多参数函数拆成链式单参数函数 |
| For-Comprehension | for 推导式 | flatMap/map/filter 的语法糖 |
| Implicit | 隐式 | 编译器自动填充的参数或转换 |
| Sealed | 密封的 | 限制子类必须在同文件定义 |
| ADT | 代数数据类型 | sealed trait + case class 的组合 |
| Option | 可选值 | Some(x) 或 None，替代 null |
| Either | 二选一 | Left(error) 或 Right(value) |
| Future | 异步计算 | 异步操作的容器，支持链式组合 |
| Tail Recursion | 尾递归 | 递归调用在最后，可优化为循环 |
| Type Inference | 类型推断 | 编译器自动推导类型 |
| Companion Object | 伴生对象 | 与 class 同名的 object，存放工厂方法和静态成员 |
| Encoder | 编码器 | Spark 中 JVM 对象与 Tungsten 二进制格式的桥梁 |
| Structural Equality | 结构相等 | 按字段值比较（`==`） |
| Referential Equality | 引用相等 | 按内存地址比较（`eq`） |

---



## 15. 面试答题策略

### 什么时候展示 Scala？

- JD 明确要求 Scala → 全程用 Scala 答题
- JD 写 "Python or Scala" → 选你更熟的，但提一句 "I'm also comfortable with Scala for production Spark pipelines"
- 纯 Python 岗 → 不需要展示 Scala，但在讨论 Spark internals 时可以提 "under the hood this is a Scala API"

### 答题风格

```text
1. 先说思路（用自然语言）
   "I'd approach this with a groupBy followed by aggregation..."

2. 写代码时用 idiomatic Scala
   - 优先 val 不用 var
   - 优先 pattern matching 不用 if-else chain
   - 优先集合操作链不用 mutable loop
   - 用 Option 不用 null check

3. 解释 tradeoff
   "This is type-safe at compile time, but the lambda inside
    map is opaque to Catalyst, so for pure SQL-expressible logic
    I'd use the DataFrame API instead."
```
